# YOLO26 试卷题目分割训练

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/

In [ ]:
!pip install ultralytics -q

In [ ]:
!cp /content/drive/MyDrive/exam_dataset_colab.zip /content/
!unzip -q /content/exam_dataset_colab.zip -d /content/
!ls -la /content/exam_dataset/

In [ ]:
yaml_content = """
path: /content/exam_dataset
train: images/train
val: images/val

names:
  0: question
"""

with open('/content/exam_dataset.yaml', 'w') as f:
    f.write(yaml_content)


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n-seg.pt')

results = model.train(
    data='/content/exam_dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='exam_segment',
    device=0,
    patience=20,
    save=True,
    plots=True,
)


In [ ]:
#test
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import cv2

model = YOLO('runs/segment/exam_segment/weights/best.pt')

import os
val_images = '/content/exam_dataset/images/val'
if os.path.exists(val_images):
    test_images = [f for f in os.listdir(val_images) if f.endswith('.jpg')]
    if test_images:
        test_image = os.path.join(val_images, test_images[0])
        results = model(test_image, conf=0.25)
        for result in results:
            annotated = result.plot()
            cv2_imshow(annotated)
        print(f"检测到 {len(results[0].masks) if results[0].masks else 0} 个题目区域")

In [ ]:
import shutil

shutil.copy('runs/segment/exam_segment/weights/best.pt', '/content/drive/MyDrive/exam_best.pt')

shutil.make_archive('/content/drive/MyDrive/exam_train_results', 'zip', 'runs/segment/exam_segment')